# Implicit-step derivation for multi-layer Coriolis + vertical diffusion

This notebook unpacks, in fine-grained detail, the algebra used in the *Implicit-Coriolis and drag* and *Algorithm* cells of `Equations-and-algorithm.ipynb`. Once the explicit accelerations $\dot u_k$ and $\dot v_k$ at time $n$ have been formed, we determine $\Delta u_k = u_k^{n+1} - u_k^n$ and $\Delta v_k = v_k^{n+1} - v_k^n$ from the $K$-element coupled system

\begin{align}
\left( \frac{1}{\Delta t} I + \alpha_\nu L \right) \Delta u - f \alpha_f \, \Delta v &= \dot u \\
\left( \frac{1}{\Delta t} I + \alpha_\nu L \right) \Delta v + f \alpha_f \, \Delta u &= \dot v
\end{align}

at each grid column. Here $L$ is the per-column tridiagonal vertical-diffusion operator built from the interfacial-stress coefficients $a_{k\pm 1/2}$ (with $a_{1/2}=0$, $a_{k-1/2}=2\nu_v/(h_{k-1}+h_k)$ for interior interfaces, and $a_{K+1/2}=\epsilon$), $f$ is the Coriolis frequency at the column, $\alpha_\nu, \alpha_f \in [0,1]$ are time-weighting parameters, and $I$ is the $K\times K$ identity. The unknowns $\Delta u$ and $\Delta v$ are $K$-vectors.

We take three steps to get from this starting point to the implementation in `_step_numba`:

1. **Algebraic elimination (Steps 2–3)** produces a pentadiagonal equation for $\Delta u$ alone (and for $\Delta v$ alone), useful for checking the $K=1$ limit against the single-layer closed form.
2. **A complex-variable trick (Steps 4–5)** block-diagonalises the same system into one complex tridiagonal — same answer, about half the bandwidth.
3. **Row-scaling and TDMAH2 (Steps 7–8)** then symmetrize the tridiagonal and remove the $1/h_k$ factors so the resulting cancellation-free Thomas variant is well-conditioned as layers vanish.

The first two steps are algebra; the third is what makes the scheme robust in practice.

## Step 1 — normalise by $\Delta t$ and abbreviate

Multiply both equations by $\Delta t$:

\begin{align}
\left( I + \alpha_\nu \Delta t \, L \right) \Delta u - \alpha_f \Delta t \, f \, \Delta v &= \Delta t \, \dot u \\
\left( I + \alpha_\nu \Delta t \, L \right) \Delta v + \alpha_f \Delta t \, f \, \Delta u &= \Delta t \, \dot v
\end{align}

Now bundle the constant pieces into shorthand symbols:

\begin{align}
A &\equiv I + \alpha_\nu \Delta t \, L && \text{(real, } K \times K \text{, tridiagonal in } k \text{)} \\
c &\equiv \alpha_f \Delta t \, f && \text{(real scalar at this column)} \\
r_u &\equiv \Delta t \, \dot u, \qquad r_v \equiv \Delta t \, \dot v && \text{(known } K\text{-vectors)}
\end{align}

so that the system simplifies to

\begin{align}
A \, \Delta u - c \, \Delta v &= r_u && (1) \\
A \, \Delta v + c \, \Delta u &= r_v && (2)
\end{align}

This is the form we work with from here on. $A$, $c$, $r_u$, $r_v$ are all known; $\Delta u, \Delta v$ are the unknowns.

A useful structural observation before we start: the same operator $A$ appears on the left of both equations, and the same scalar $c$ multiplies the cross-coupling between them. This is what makes block-diagonalisation possible later.

## Step 2 — Approach A: algebraic elimination of $\Delta v$

To find $\Delta u$ alone, we eliminate $\Delta v$ from (1).

**Step 2a.** Solve (2) for $A \, \Delta v$:

$$ A \, \Delta v = r_v - c \, \Delta u. \qquad (2') $$

**Step 2b.** Apply $A$ to both sides of (1):

$$ A (A \, \Delta u) - c (A \, \Delta v) = A \, r_u $$
$$ A^2 \, \Delta u - c \, A \, \Delta v = A \, r_u. \qquad (1') $$

(We can apply $A$ this way because $A$ is a fixed linear operator and $\Delta u$, $\Delta v$ are vectors; matrix-vector products are associative.)

**Step 2c.** Substitute $A \, \Delta v$ from $(2')$ into $(1')$:

$$ A^2 \, \Delta u - c \, (r_v - c \, \Delta u) = A \, r_u $$
$$ A^2 \, \Delta u - c \, r_v + c^2 \, \Delta u = A \, r_u $$
$$ A^2 \, \Delta u + c^2 \, \Delta u = A \, r_u + c \, r_v $$

Group the $\Delta u$ terms (using $c^2 \Delta u = c^2 I \, \Delta u$ since $\Delta u$ is a vector and $I$ is the identity):

$$ \boxed{ \left( A^2 + c^2 \, I \right) \Delta u = A \, r_u + c \, r_v. } \qquad (*) $$

**Step 2d.** The Δv equation follows by symmetry. From (1) solve for $A \Delta u$:

$$ A \, \Delta u = r_u + c \, \Delta v. \qquad (1'') $$

Apply $A$ to (2):

$$ A^2 \, \Delta v + c \, A \, \Delta u = A \, r_v $$

Substitute $(1'')$:

$$ A^2 \, \Delta v + c \, (r_u + c \, \Delta v) = A \, r_v $$
$$ A^2 \, \Delta v + c^2 \, \Delta v = A \, r_v - c \, r_u $$
$$ \boxed{ \left( A^2 + c^2 \, I \right) \Delta v = A \, r_v - c \, r_u. } \qquad (**) $$

The LHS operator is the same in $(*)$ and $(**)$ — a key point for both efficiency and for the equivalence proof below.

## Step 3 — what $A^2 + c^2 I$ looks like, and the $K=1$ limit

Because $A$ is tridiagonal in $k$, $A^2$ is *pentadiagonal*: $(A^2)_{k, k\pm 2}$ is non-zero. So $A^2 + c^2 I$ is also pentadiagonal in $k$, with bandwidth 5. Solving a pentadiagonal $K \times K$ system by direct elimination takes ~$9K$ floating-point operations versus ~$5K$ for a tridiagonal, i.e. roughly twice the work. That motivates Approach B (next).

But first, $(*)$ has one clean special case: **$K = 1$**. The operator $L$ then has just a bottom-interface entry, $L = \varepsilon/h$, so

$$ A = 1 + \frac{\alpha_\nu \, \Delta t \, \varepsilon}{h}, \qquad A^2 + c^2 I = A^2 + c^2 $$

are scalars and $(*)$ degenerates to the scalar equation

$$ (A^2 + c^2) \, \Delta u = A \, r_u + c \, r_v $$

$$ \Delta u = \frac{A \, r_u + c \, r_v}{A^2 + c^2} = \frac{\Delta t \, [\, A \, \dot u + c \, \dot v \,]}{A^2 + c^2}. $$

Substituting back the long names:

$$ \Delta u = \frac{\Delta t \, \left[ \left(1 + \frac{\Delta t \alpha_\nu \varepsilon}{h}\right) \dot u + \alpha_f \Delta t \, f \, \dot v \right]}{\left(1 + \frac{\Delta t \alpha_\nu \varepsilon}{h}\right)^2 + \alpha_f^2 \Delta t^2 f^2}. $$

This is exactly the single-layer closed form given in the *Implicit-Coriolis and drag* cell of the main notebook, with $\alpha_\nu \leftrightarrow \alpha_\epsilon$ (they label the same parameter).

## Step 4 — Approach B: block-diagonalisation via a complex variable

Define a complex unknown and complex right-hand side by stacking $\Delta u$ and $\Delta v$ (or $r_u$ and $r_v$) as the real and imaginary parts:

$$ \Delta w \equiv \Delta u + i \, \Delta v, \qquad r_w \equiv r_u + i \, r_v. $$

(Both are $K$-vectors of complex numbers.)

**Step 4a.** Form the linear combination "(1) plus $i$ times (2)":

$$ \underbrace{A \, \Delta u - c \, \Delta v}_{\text{from (1)}} + i \, \underbrace{(A \, \Delta v + c \, \Delta u)}_{\text{from (2)}} = r_u + i \, r_v $$

**Step 4b.** Group terms by what they multiply:

\begin{align}
A \, \Delta u - c \, \Delta v + i \, A \, \Delta v + i \, c \, \Delta u
&= A \, (\Delta u + i \, \Delta v) + c \, (-\Delta v + i \, \Delta u) \\
&= A \, \Delta w + c \, (-\Delta v + i \, \Delta u)
\end{align}

**Step 4c.** The trick is that the second bracket is also $\Delta w$, multiplied by $i$:

$$ i \, \Delta w = i \, (\Delta u + i \, \Delta v) = i \, \Delta u + i^2 \, \Delta v = -\Delta v + i \, \Delta u. \qquad \checkmark $$

So $-\Delta v + i \, \Delta u = i \, \Delta w$, and therefore $c \, (-\Delta v + i \, \Delta u) = i c \, \Delta w$.

**Step 4d.** Substituting back:

$$ A \, \Delta w + i \, c \, \Delta w = r_w $$
$$ \boxed{ (A + i \, c \, I) \, \Delta w = r_w. } \qquad (\star) $$

This is the punch line. The operator $A + ic\, I$ adds the *same imaginary scalar* $ic$ to every diagonal entry of the real tridiagonal $A$ — so $A + ic\, I$ is **complex tridiagonal in $k$**. The bandwidth is 3 instead of 5: the same as the original $A$.

Solve $(\star)$ once per column by the standard Thomas algorithm using complex arithmetic, then read off

$$ \Delta u = \mathrm{Re}(\Delta w), \qquad \Delta v = \mathrm{Im}(\Delta w). $$

Each complex add/multiply/divide is roughly $2\times$ the cost of the real one, so the complex-tridiagonal solve is about $2 \cdot 5K = 10K$ flops versus $9K$ for the real pentadiagonal — roughly comparable in count, but with much simpler code (the existing Thomas algorithm with `complex128` arrays).

## Step 5 — Approach B implies Approach A (and they agree)

Multiplying $(\star)$ by the conjugate operator $A - ic\, I$:

\begin{align}
(A - i c \, I)(A + i c \, I) \, \Delta w &= (A - i c \, I) \, r_w
\end{align}

The left-hand side simplifies using $A \, (i c \, I) = (i c \, I) \, A = i c \, A$ (a scalar commutes with everything) and $(ic)(ic) = i^2 c^2 = -c^2$:

$$ (A - i c \, I)(A + i c \, I) = A^2 + i c \, A - i c \, A - i^2 c^2 \, I = A^2 + c^2 \, I. $$

Expanding the right-hand side:

$$ (A - i c \, I)(r_u + i r_v) = A \, r_u + i \, A \, r_v - i c \, r_u + c \, r_v = (A r_u + c r_v) + i (A r_v - c r_u). $$

So

$$ (A^2 + c^2 \, I) \, (\Delta u + i \, \Delta v) = (A r_u + c r_v) + i (A r_v - c r_u). $$

Equating real and imaginary parts:

\begin{align}
(A^2 + c^2 \, I) \, \Delta u &= A \, r_u + c \, r_v && \checkmark \, \text{matches } (*) \\
(A^2 + c^2 \, I) \, \Delta v &= A \, r_v - c \, r_u && \checkmark \, \text{matches } (**)
\end{align}

So the complex-tridiagonal solve $(\star)$ is just a more efficient way of solving the same pentadiagonal pair $(*)$, $(**)$. They give bit-equivalent $\Delta u, \Delta v$ (up to round-off in the different operation order).

## Step 6 — spatial discretisation on the C-grid

The algebra above was written as if $\Delta u$ and $\Delta v$ live at the same point. On an Arakawa C-grid they do not: $\Delta u$ is defined at u-points, $\Delta v$ at v-points. When we write the discrete equations at a u-point, the "$\Delta v$" appearing in (1) and (2) must be interpreted as $\Delta v$ averaged from the surrounding v-points to the u-point — write this as $\overline{\Delta v}^{ij}$. Likewise $\dot v$ becomes $\overline{\dot v}^{ij}$.

The discrete system *at a u-point* is therefore

\begin{align}
A \, \Delta u - c \, \overline{\Delta v}^{ij} &= \Delta t \, \dot u \\
A \, \overline{\Delta v}^{ij} + c \, \Delta u &= \Delta t \, \overline{\dot v}^{ij}
\end{align}

with the same $A$ and $c$ as before (built from $h_k, f$ at this u-point). Applying $(\star)$:

$$ \Delta w \big|_{\text{u-pt}} = \Delta u + i \, \overline{\Delta v}^{ij}, \quad r_w \big|_{\text{u-pt}} = \Delta t \, \dot u + i \, \Delta t \, \overline{\dot v}^{ij}, $$

solve $(A + ic \, I) \, \Delta w = r_w$, and take

$$ \Delta u = \mathrm{Re}(\Delta w). $$

The imaginary part is $\overline{\Delta v}^{ij}$ — an *interpolated* estimate of $\Delta v$ at the u-point that we discard, because the prognostic $\Delta v$ lives at v-points.

To obtain $\Delta v$, repeat the exercise at v-points with $A$, $c$ built from $h_k, f$ at the v-point and the interpolated $\overline{\dot u}^{ij}$ replacing $\dot u$:

$$ \Delta v = \mathrm{Im}\!\left[ (A + ic \, I)^{-1} (\Delta t \, \overline{\dot u}^{ij} + i \, \Delta t \, \dot v) \right] $$

(at the v-point). The two solves are independent, so they can be performed sequentially or in parallel. This is the C-grid semi-implicit Coriolis treatment generalised to multi-layer with implicit vertical diffusion.

## Step 7 — concrete entries, and row-scaling for symmetry and conditioning

Filling in $A = I + \alpha_\nu \Delta t\, L$ and $c = \alpha_f \Delta t\, f$ at a u-point gives the matrix elements of $A + ic\, I$:

\begin{align}
(A + ic \, I)_{k, k-1} &= -\, \alpha_\nu \Delta t \, \frac{a_{k-1/2}}{h_k^u} && (k \ge 1, \text{ sub-diagonal}) \\
(A + ic \, I)_{k, k}   &= 1 + \alpha_\nu \Delta t \, \frac{a_{k-1/2} + a_{k+1/2}}{h_k^u} \;+\; i \alpha_f \Delta t \, f^u && (\text{diagonal}) \\
(A + ic \, I)_{k, k+1} &= -\, \alpha_\nu \Delta t \, \frac{a_{k+1/2}}{h_k^u} && (k \le K-2, \text{ super-diagonal})
\end{align}

with the boundary conventions $a_{1/2} = 0$ at the surface (wind goes into $\dot u$ explicitly) and $a_{K+1/2} = \varepsilon$ at the bottom (so the top row has no sub-diagonal and the bottom row picks up the drag term on its diagonal but has no super-diagonal). The right-hand side is

$$ r_w \big|_k = \Delta t\, \dot u_k + i\, \Delta t\, \overline{\dot v}^{ij}_k. $$

**Two issues with this natural form.** Compare the entries straddling row $k$ and $k+1$:

$$ (A + ic\, I)_{k, k+1} = -\alpha_\nu \Delta t \, \frac{a_{k+1/2}}{h_k^u}, \qquad (A + ic\, I)_{k+1, k} = -\alpha_\nu \Delta t \, \frac{a_{k+1/2}}{h_{k+1}^u} $$

They differ by the ratio $h_k^u / h_{k+1}^u$, so the matrix is **asymmetric** unless all layers are equally thick. And every coefficient carries a factor of $1/h_k^u$ — so **as $h_k^u \to 0$ the entries diverge**, and the standard Thomas pivot $m_k = a^\text{sub}_k / b'_{k-1}$ would be followed by a subtraction $b_k - m_k\, c^\text{sup}_{k-1}$ in which both terms can be near-equal positives. That is exactly the kind of catastrophic cancellation we'd like to avoid in models that allow vanishing layers.

**Row-scaling fix.** Both issues disappear if we multiply equation $k$ by $h_k^u$ before solving (this is just a re-scaling of an individual linear equation — it doesn't change $\Delta w$). Define the **scaled** interfacial-stress coefficients

$$ a^*_{k\pm 1/2} \equiv \alpha_\nu \Delta t \, a_{k\pm 1/2} \qquad (\text{real}, \ge 0) $$

with boundary values $a^*_{1/2} = 0$ at the surface and $a^*_{K+1/2} = \alpha_\nu \Delta t\, \varepsilon$ at the bottom. After row-scaling, the system $\tilde A_{ic}\, \Delta w = y$ has entries

\begin{align}
\tilde A_{ic;\, k, k-1} &= -\, a^*_{k-1/2} && (k \ge 1, \text{ sub-diagonal}) \\
\tilde A_{ic;\, k, k}   &= h_k^u (1 + i\, c) + a^*_{k-1/2} + a^*_{k+1/2} && (\text{diagonal}) \\
\tilde A_{ic;\, k, k+1} &= -\, a^*_{k+1/2} && (k \le K-2, \text{ super-diagonal})
\end{align}

with right-hand side

$$ y_k = h_k^u \, r_w \big|_k = h_k^u\, \Delta t\, ( \dot u_k + i\, \overline{\dot v}^{ij}_k ). $$

The off-diagonals are now **symmetric** ($\tilde A_{ic;\, k+1, k} = -a^*_{k+1/2} = \tilde A_{ic;\, k, k+1}$) and **real**, and no entry contains $1/h_k^u$ any more.

**Vanishing-layer behaviour after row-scaling.** As $h_k^u \to 0$, the diagonal stays finite,

$$ \tilde A_{ic;\, k, k} \;\to\; a^*_{k-1/2} + a^*_{k+1/2}, $$

and the RHS $y_k = h_k^u\, r_w \big|_k \to 0$ at the same rate. The $k$-th equation locally reduces to $a^*_{k-1/2}(\Delta w_k - \Delta w_{k-1}) + a^*_{k+1/2}(\Delta w_k - \Delta w_{k+1}) = 0$, i.e. $\Delta w_k$ is a weighted average of its neighbours — the physically correct behaviour for a vanishing layer that is "swept along" by adjacent layers. The algorithm doesn't need any `+ hsub` regularisation; it just works.

The v-point version is identical with $h_k^u \to h_k^v$, $f^u \to f^v$, and $y_k = h_k^v\, \Delta t\, (\overline{\dot u}^{ij}_k + i\, \dot v_k)$.

## Step 8 — TDMAH2: cancellation-free Thomas on the row-scaled system

We solve $\tilde A_{ic}\, \Delta w = y$ with Hallberg's cancellation-free variant of Thomas. The idea is to track positive intermediate quantities $q_k \equiv -c'_k$ and $Q_k \equiv 1 - q_k$ in place of the standard Thomas' modified diagonals — for the symmetric system written in Step 7 this rearranges the bookkeeping so that no subtraction of nearly-equal positive numbers appears in any forward-sweep denominator. The recurrence carries the substitution $h_k \to h_k(1+ic)$ transparently because $c$ enters only as an additive imaginary on the diagonal $h_k$ part — the $a^*_{k\pm 1/2}$ stay real and non-negative, and the algorithm is structurally identical to the real case.

**Forward sweep.** Initialise at $k = 0$:

\begin{align}
\beta_0 &= 1\, /\, \big( h_0^u\, (1 + ic) + a^*_{1/2} + a^*_{3/2} \big) \\
q_0     &= a^*_{3/2}\, \beta_0 \\
Q_0     &= h_0^u\, (1 + ic)\, \beta_0 \\
y'_0    &= h_0^u\, \Delta t\, (\dot u_0 + i\, \overline{\dot v}^{ij}_0)\, \beta_0
\end{align}

then for $k = 1, \dots, K-1$:

\begin{align}
\beta_k &= 1\, /\, \big( h_k^u\, (1 + ic) + a^*_{k-1/2}\, Q_{k-1} + a^*_{k+1/2} \big) \\
q_k     &= a^*_{k+1/2}\, \beta_k \\
Q_k     &= \big( h_k^u\, (1 + ic) + a^*_{k-1/2}\, Q_{k-1} \big)\, \beta_k \\
y'_k    &= \big( h_k^u\, \Delta t\, (\dot u_k + i\, \overline{\dot v}^{ij}_k) + a^*_{k-1/2}\, y'_{k-1} \big)\, \beta_k
\end{align}

Every denominator $\beta_k^{-1}$ is a sum of three pieces, all real-non-negative or complex-with-real-non-negative-part: $a^*_{k-1/2}\, Q_{k-1}$ (with $\mathrm{Re}(Q_{k-1}) \ge 0$ by induction from the recurrence), $a^*_{k+1/2}$ (real $\ge 0$), and $h_k^u (1+ic)$ (real part $h_k^u \ge 0$). No subtraction of nearly-equal positives appears anywhere in the forward sweep. The real part of $\beta_k^{-1}$ is bounded below by $h_k^u + a^*_{k+1/2}$, so the inverse is well-defined even as individual layers vanish.

**Back substitution.** From the bottom up:

\begin{align}
\Delta w_{K-1} &= y'_{K-1} \\
\Delta w_k     &= y'_k + q_k\, \Delta w_{k+1} \qquad (k = K-2, \dots, 0)
\end{align}

Note the `+` rather than `-`: that's the consequence of defining $q_k \equiv -c'_k$ (where $c'_k$ would be the standard Thomas' modified super-diagonal entry), absorbing the negative sign of the off-diagonal $\tilde A_{ic;\, k, k+1} = -a^*_{k+1/2}$ into the definition of $q$.

**Apply.** At u-points, the new layer velocities are

$$ u_k^{n+1} = u_k^n + \mathrm{Re}(\Delta w_k); $$

at v-points (analogous solve using $h_k^v$, $f^v$, and RHS $h_k^v\, \Delta t\, (\overline{\dot u}^{ij}_k + i\, \dot v_k)$) the new velocities are

$$ v_k^{n+1} = v_k^n + \mathrm{Im}(\Delta w_k). $$

**$K = 1$ check.** Only the $k = 0$ initialisation runs. With $a^*_{1/2} = 0$ and $a^*_{3/2} = \alpha_\nu \Delta t\, \varepsilon$ we get

$$ \Delta w_0 = y'_0 = \frac{h_0^u\, \Delta t\, (\dot u + i\, \overline{\dot v})}{h_0^u\, (1 + ic) + \alpha_\nu \Delta t\, \varepsilon}. $$

Dividing top and bottom by $h_0^u$,

$$ \Delta w_0 = \frac{\Delta t\, (\dot u + i\, \overline{\dot v})}{(1 + \alpha_\nu \Delta t\, \varepsilon / h_0^u) + ic}, $$

whose real part is exactly the single-layer closed form $\Delta u = \Delta t\, [(1 + \alpha_\nu \Delta t\, \varepsilon / h)\, \dot u + \alpha_f \Delta t\, f\, \dot v]\, /\, [(1 + \alpha_\nu \Delta t\, \varepsilon / h)^2 + \alpha_f^2 \Delta t^2 f^2]$ from Step 3.

**Implementation note.** In `_step_numba` (`OSSWEM.py`) the recurrence above is implemented as a $k$-loop in which every right-hand side is a 2D `(nj, ni)` complex array operation, so the column problems are solved simultaneously across all $(i, j)$. Only $q$ and $y'$ need to be stored as 3D arrays — $Q$ and $\beta$ are rolling 2D variables — giving slightly lower memory pressure than a buffered standard Thomas (4 complex 3D arrays $\to$ 2).